In [7]:
def read_csv(filename):
    file = open(filename, "r")
    lines = file.readlines()
    file.close()
    headers = lines[0].strip().split(",")
    data = []
    for line in lines[1:]:
        row = line.strip().split(",")
        data.append(row)
    return headers, data
def count_classes(data):
    counts = {}
    for row in data:
        label = row[-1]
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    return counts
def entropy(data):
    counts = count_classes(data)
    total = len(data)
    ent = 0
    for key in counts:
        p = counts[key] / float(total)
        log_value = 0
        temp = p
        while temp < 1:
            temp *= 2
            log_value -= 1
        ent -= p * log_value
    return ent
def split_data(data, attribute, value):
    subset = []
    for row in data:
        if row[attribute] == value:
            subset.append(row)
    return subset
def unique_values(data, attribute):
    values = []
    for row in data:
        if row[attribute] not in values:
            values.append(row[attribute])
    return values
def information_gain(data, attribute):
    total_entropy = entropy(data)
    values = unique_values(data, attribute)
    weighted_entropy = 0
    for value in values:
        subset = split_data(
            data,
            attribute,
            value
        )
        probability = len(subset) / float(len(data))
        weighted_entropy += probability * entropy(subset)
    return total_entropy - weighted_entropy
def best_attribute(data, attributes):
    max_gain = -1
    best = -1
    for attr in attributes:
        gain = information_gain(
            data,
            attr
        )
        if gain > max_gain:
            max_gain = gain
            best = attr
    return best
def majority_class(data):
    counts = count_classes(data)
    max_value = -1
    result = None
    for key in counts:
        if counts[key] > max_value:
            max_value = counts[key]
            result = key
    return result
class Node:
    def __init__(self, label):
        self.label = label
        self.children = {}
def generate_tree(data, attributes):
    labels = []
    for row in data:
        labels.append(row[-1])
    if labels.count(labels[0]) == len(labels):
        return Node(labels[0])
    if len(attributes) == 0:
        return Node(
            majority_class(data)
        )
    split_attribute = best_attribute(
        data,
        attributes
    )
    node = Node(split_attribute)
    values = unique_values(
        data,
        split_attribute
    )
    remaining = []
    for a in attributes:
        if a != split_attribute:
            remaining.append(a)
    for value in values:
        subset = split_data(
            data,
            split_attribute,
            value
        )
        if len(subset) == 0:
            node.children[value] = Node(
                majority_class(data)
            )
        else:
            node.children[value] = generate_tree(
                subset,
                remaining
            )
    return node
def print_tree(node, headers, space=""):
    if len(node.children) == 0:
        print(space + "Class : " + str(node.label))
    else:
        print(space + "Split Attribute : " 
              + headers[node.label])
        for value in node.children:
            print(space + " Value -> " + value)
            print_tree(
                node.children[value],
                headers,
                space+"   " )
filename = "shop (1).csv"
headers, data = read_csv(filename)
attributes = []
for i in range(len(headers)-1):
    attributes.append(i)
tree = generate_tree(data,attributes)
print_tree(tree, headers)

Split Attribute : age
 Value -> youth
   Split Attribute : student
    Value -> no
      Class : no
    Value -> yes
      Class : yes
 Value -> middleage
   Class : yes
 Value -> senior
   Split Attribute : creditrating
    Value -> fair
      Class : yes
    Value -> excellent
      Class : no
